# Study 917 — Stale NAV 🕰️

**Wall Street closes strong. Tokyo has been shut for hours. Does the Japan ETF owe you that
move tomorrow?**

Five US-listed single-country ETFs — **EWJ** (Japan), **EWG** (Germany), **FXI** (China/HK),
**EWA** (Australia), **EWU** (UK) — track markets that are *closed* while New York trades.
The textbook says their prices go stale with respect to the US session and must catch up
next day. We test it on daily **total-return** closes, 1996-03-18 → 2026-06-30
(SPY back to 1993-01-29, 8,411 days), one execution lag, 10 bps one-way cost,
every leg **excess-of-cash**.

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `42e35a0143ca`); the one
live cell runs the fast offline synthetic control. As-of 2026-06-30.*


## 1. Why anyone believes this

It used to be true, and people went to court over it. In the 1990s an international **mutual fund** struck one price a day, off the last local closes. If the S&P had roared while Tokyo slept, you could buy that fund at yesterday's Tokyo prices and collect the catch-up the next morning. Regulators killed it with *fair-value pricing* after the 2003 market-timing scandal.

The modern retail version points at country **ETFs** instead. Same intuition, different wrapper — and the wrapper is exactly what matters, because an ETF trades in New York all day, right next to the S&P it is supposedly ignoring.

## 2. What the tape says: the sign is backwards

Regress each fund's *next* day return on today's SPY return. Catch-up means a **positive** slope. All five come out **negative** — the funds hand part of the US move *back*.

In [1]:
R = dict(ewj=(-0.111, -5.85, -0.027, -1.51), ewg=(-0.06, -2.25, 0.024, 1.58), fxi=(-0.263, -6.54, -0.16, -4.3), ewa=(-0.073, -1.75, 0.011, 0.45), ewu=(-0.056, -2.13, 0.028, 2.01), dom_beta=-0.082, dom_t=-3.51)
for name, v in [('EWJ Japan', R['ewj']), ('EWG Germany', R['ewg']),
                ('FXI China/HK', R['fxi']), ('EWA Australia', R['ewa']),
                ('EWU UK', R['ewu'])]:
    print('%-14s next-day slope on SPY: %+.3f  (t = %+.2f)' % (name, v[0], v[1]))
print()
print('and SPY on ITSELF            : %+.3f  (t = %+.2f)  <- the catch'
      % (R['dom_beta'], R['dom_t']))

EWJ Japan      next-day slope on SPY: -0.111  (t = -5.85)
EWG Germany    next-day slope on SPY: -0.060  (t = -2.25)
FXI China/HK   next-day slope on SPY: -0.263  (t = -6.54)
EWA Australia  next-day slope on SPY: -0.073  (t = -1.75)
EWU UK         next-day slope on SPY: -0.056  (t = -2.13)

and SPY on ITSELF            : -0.082  (t = -3.51)  <- the catch


## 3. The catch — the US market does this to *itself*

Look at the last line. After a strong day, **SPY itself** gives a little back the next day (-0.082, *t* = -3.51). A country ETF moves roughly one-for-one with the US market, so it inherits that wobble for free. It has nothing to do with Tokyo being shut.

Strip it out — measure each fund *against SPY on the same day* — and the timezone-specific effect essentially disappears: EWJ -0.027 (*t* = -1.51), EWG +0.024 (+1.58), EWA +0.011 (+0.45), EWU +0.028 (+2.01). Only FXI stands out, at -0.160 — still the wrong sign, and (next section) only in its youth.

> ⚠️ **Careful, and this matters.** "Measure it against SPY" quietly assumes each fund moves *one-for-one* with the US market. They do not — EWJ moves about 0.75 for every 1 of SPY, FXI about 1.16. Redo the subtraction at each fund's real ratio and the picture gets **worse for the story, not better**: EWJ's slope goes from *t* = -1.51 to -3.19 — significant, and still **negative**. Correcting the arithmetic finds more give-back, never more catch-up. Not one fund comes out positive: the best is EWG at *t* = +1.54.

> 🔬 **For the quants** — with five funds tested the family-wise 5% bar is |*t*| ≥ 2.58, not 1.96. Nothing pointing the claimed way is anywhere near it, under either subtraction.

## 4. Try to trade it anyway

The rule: after a **top-decile** SPY day, own an equal-weight basket of all five funds for the next day; sit in T-bills otherwise. That is 819 trading days (10.7% of the tape), 1,402 position changes, and two spreads per round trip.

In [2]:
R = dict(on_bps=-4.92, on_t=-0.82, off_bps=3.45, rel_bps=-1.3, rel_t=-0.39,
         long_sh=-0.701, bh_sh=0.298, boot_lo=-19.75, boot_hi=6.63)
print('day after a top-decile SPY day : %+.2f bps   (t = %+.2f)'
      % (R['on_bps'], R['on_t']))
print('every other day                : %+.2f bps' % R['off_bps'])
print('...and net of what SPY did that same day: %+.2f bps (t = %+.2f)'
      % (R['rel_bps'], R['rel_t']))
print()
print('the rule, net of 10 bps  : excess Sharpe %+.3f' % R['long_sh'])
print('just holding the basket  : excess Sharpe %+.3f' % R['bh_sh'])
print('bootstrap 95%% CI on the trigger-day return: [%+.2f, %+.2f] bps'
      % (R['boot_lo'], R['boot_hi']))

day after a top-decile SPY day : -4.92 bps   (t = -0.82)
every other day                : +3.45 bps
...and net of what SPY did that same day: -1.30 bps (t = -0.39)

the rule, net of 10 bps  : excess Sharpe -0.701
just holding the basket  : excess Sharpe +0.298
bootstrap 95% CI on the trigger-day return: [-19.75, +6.63] bps


## 5. Three ways it fails, for good measure

- **Flip the trade.** If the funds *reverse*, shorting them should pay. Gross it is barely positive (excess Sharpe +0.127, *t* = +0.70 — a coin flip), and it is under water by 5 bps one-way (-0.146) *before* you pay anyone to borrow the shares.
- **Wait one more day.** The base rule assumes you can buy at the very close that defines the signal. Wait a single extra day and the trigger-day return is **+0.70 bps (*t* = +0.13)** — gone.
- **Split the history.** 1996–2009: +3.95 bps (*t* = +0.53). 2010–2026: -13.17 bps (-1.58). Opposite signs, neither significant.

## 6. Live check — the machinery *can* find a catch-up (offline synthetic)

Before believing a null, check the detector. We build a fake world where yesterday's US move genuinely leaks into today's country return, and a null world where the funds are just as correlated with the US *today* but owe nothing tomorrow. The same code must fire on one and stay silent on the other.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from stale_nav import data, strategy as st
p1, t1 = data.synthetic_panel(signal_strength=1.0, seed=917)
p0, t0 = data.synthetic_panel(signal_strength=0.0, seed=917)
d1, d0 = st.synthetic_detect(p1, t1), st.synthetic_detect(p0, t0)
print('planted catch-up of %+.3f -> recovered %+.3f (t %+.1f), trigger day %+.1f bps'
      % (d1['planted_beta'], d1['beta_mean'], d1['t_beta_mean'], d1['mean_on_bps']))
print('no catch-up planted    -> recovered %+.3f (t %+.1f), trigger day %+.1f bps'
      % (d0['beta_mean'], d0['t_beta_mean'], d0['mean_on_bps']))

planted catch-up of +0.250 -> recovered +0.249 (t +18.0), trigger day +53.6 bps
no catch-up planted    -> recovered -0.001 (t -0.1), trigger day +5.3 bps


## Verdict

- **Signal — None.** There is no catch-up: every slope has the wrong sign, and the part that is *not* just the US market's own next-day wobble is **-1.30 bps/day (*t* = -0.39)** — -1.55 (-0.46) if you subtract SPY at the basket's real ratio of 0.906 instead of 1 — with a bootstrap interval straddling zero, opposite signs in the two eras, and nothing left if you wait one more day to trade.
- **Tradability — Mirage.** The rule loses money as stated (excess Sharpe -0.701 against +0.298 for simply holding the same five funds). The mirror short is a gross coin flip that dies at 5 bps, before borrow.
- **The honest footnote.** The 1990s evidence was real — against a once-a-day *mutual fund* NAV. An ETF trades in New York all session long, so the US move is already in the price by the close. The trade did not decay; its victim was redesigned.